In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

RESULTS_DIR = Path("../results")
OUTPUT_PNG = Path("test_metrics_boxplots.png")

SEEDS = range(120, 140)
MODELS = {
    "baseline": "Baseline",
    "tpl": "TPL",
}

METRICS = [
    ("test_loss", "Test loss"),
    ("strict_syllable_acc", "Syllable accuracy"),
    ("strict_exact_acc", "Exact accuracy"),
]


In [ ]:
rows = []

for model, label in MODELS.items():
    for seed in SEEDS:
        csv_path = RESULTS_DIR / model / f"seed_{seed}" / "test.csv"
        if not csv_path.exists():
            raise FileNotFoundError(f"Missing: {csv_path}")

        df = pd.read_csv(csv_path)
        missing = [col for col, _ in METRICS if col not in df.columns]
        if missing:
            raise KeyError(f"{csv_path}: missing columns {missing}")

        row = {
            "model": label,
            "seed": seed,
        }
        for col, _ in METRICS:
            row[col] = df.iloc[0][col]
        rows.append(row)

results = pd.DataFrame(rows)

print(f"Loaded {len(results)} runs")
print(results.groupby("model").size())
results.head()


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 4))

for ax, (metric, title) in zip(axes, METRICS):
    values = [
        results.loc[results["model"] == "Baseline", metric].to_numpy(),
        results.loc[results["model"] == "TPL", metric].to_numpy(),
    ]

    ax.boxplot(values, labels=["Baseline", "TPL"])
    ax.set_title(title)
    ax.grid(axis="y", alpha=0.25)

fig.tight_layout()
fig.savefig(OUTPUT_PNG, dpi=300, bbox_inches="tight")
plt.show()

print(f"Saved: {OUTPUT_PNG.resolve()}")
